In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from utils.experiments.sweeper import ParameterSweeper
from src.strategies.swing_range_expansion.runner.backtest_runner import SwingRangeExpansionBacktestRunner
from utils.reporting.analytics import calculate_additional_metrics


In [ ]:
# Define parameter grid for sweep
param_grid = {
    'risk_per_trade': [0.01, 0.02, 0.03],  # 1%, 2%, 3% risk per trade
    'profit_target_r': [2.0, 3.0, 4.0],   # R-multiples for profit target
    'stop_loss_r': [1.0, 1.5, 2.0],       # R-multiples for stop loss
}

print(f"Total combinations: {len(param_grid['risk_per_trade']) * len(param_grid['profit_target_r']) * len(param_grid['stop_loss_r'])}")
print(f"Parameter grid: {param_grid}")


In [ ]:
# Initialize the runner
runner = SwingRangeExpansionBacktestRunner()

# Create parameter sweeper
sweeper = ParameterSweeper(runner, param_grid)

# Run the sweep (this may take a few minutes)
print("Starting parameter sweep...")
results = sweeper.run_sweep(max_workers=4)
print(f"Completed {len(results)} parameter combinations")


In [ ]:
# Convert results to DataFrame
df_results = pd.DataFrame(results)

# Display basic statistics
print("Parameter Sweep Results Summary:")
print(f"Number of combinations: {len(df_results)}")
print(f"Best Total Return: {df_results['total_return'].max():.2%}")
print(f"Best Sharpe Ratio: {df_results['sharpe_ratio'].max():.3f}")
print(f"Lowest MDD: {df_results['mdd_pct'].min():.2%}")

# Show top 5 performers by total return
print("\nTop 5 by Total Return:")
top_performers = df_results.nlargest(5, 'total_return')[['risk_per_trade', 'profit_target_r', 'stop_loss_r', 'total_return', 'sharpe_ratio', 'mdd_pct']]
print(top_performers)


In [ ]:
# Create 3D scatter plot for parameter relationships
fig = go.Figure(data=go.Scatter3d(
    x=df_results['risk_per_trade'],
    y=df_results['profit_target_r'],
    z=df_results['stop_loss_r'],
    mode='markers',
    marker=dict(
        size=8,
        color=df_results['total_return'],
        colorscale='Viridis',
        colorbar=dict(title="Total Return"),
        showscale=True
    ),
    text=[f"Risk: {r:.1%}<br>PT: {pt:.1f}R<br>SL: {sl:.1f}R<br>Return: {ret:.2%}<br>Sharpe: {sr:.3f}" 
          for r, pt, sl, ret, sr in zip(df_results['risk_per_trade'], df_results['profit_target_r'], 
                                       df_results['stop_loss_r'], df_results['total_return'], df_results['sharpe_ratio'])],
    hovertemplate='%{text}<extra></extra>'
))

fig.update_layout(
    title='Parameter Sweep Results - Total Return',
    scene=dict(
        xaxis_title='Risk Per Trade',
        yaxis_title='Profit Target (R)',
        zaxis_title='Stop Loss (R)'
    ),
    width=800,
    height=600
)

fig.show()


In [ ]:
# Create heatmaps for different parameter combinations
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Total Return', 'Sharpe Ratio', 'Max Drawdown', 'Win Rate'),
    specs=[[{'type': 'heatmap'}, {'type': 'heatmap'}],
           [{'type': 'heatmap'}, {'type': 'heatmap'}]]
)

# For simplicity, fix stop_loss_r at 1.0 and show risk_per_trade vs profit_target_r
subset = df_results[df_results['stop_loss_r'] == 1.0]
pivot_return = subset.pivot(index='risk_per_trade', columns='profit_target_r', values='total_return')
pivot_sharpe = subset.pivot(index='risk_per_trade', columns='profit_target_r', values='sharpe_ratio')
pivot_mdd = subset.pivot(index='risk_per_trade', columns='profit_target_r', values='mdd_pct')
pivot_winrate = subset.pivot(index='risk_per_trade', columns='profit_target_r', values='win_rate')

fig.add_trace(go.Heatmap(z=pivot_return.values, x=pivot_return.columns, y=pivot_return.index, 
                        colorscale='RdYlGn', name='Total Return'), row=1, col=1)
fig.add_trace(go.Heatmap(z=pivot_sharpe.values, x=pivot_sharpe.columns, y=pivot_sharpe.index, 
                        colorscale='RdYlGn', name='Sharpe Ratio'), row=1, col=2)
fig.add_trace(go.Heatmap(z=pivot_mdd.values, x=pivot_mdd.columns, y=pivot_mdd.index, 
                        colorscale='RdYlGn_r', name='Max Drawdown'), row=2, col=1)
fig.add_trace(go.Heatmap(z=pivot_winrate.values, x=pivot_winrate.columns, y=pivot_winrate.index, 
                        colorscale='RdYlGn', name='Win Rate'), row=2, col=2)

fig.update_layout(title='Parameter Heatmaps (Stop Loss = 1.0R)', height=800)
fig.show()


In [ ]:
# Risk-Return scatter plot
fig = px.scatter(df_results, 
                x='mdd_pct', 
                y='total_return',
                color='sharpe_ratio',
                size='win_rate',
                hover_data=['risk_per_trade', 'profit_target_r', 'stop_loss_r'],
                title='Risk-Return Profile',
                labels={
                    'mdd_pct': 'Maximum Drawdown (%)',
                    'total_return': 'Total Return (%)',
                    'sharpe_ratio': 'Sharpe Ratio'
                })

fig.update_layout(width=800, height=600)
fig.show()

# Find the best risk-adjusted returns (highest Sharpe ratio)
best_sharpe = df_results.loc[df_results['sharpe_ratio'].idxmax()]
print(f"\nBest Risk-Adjusted Performance (Highest Sharpe Ratio):")
print(f"Parameters: Risk={best_sharpe['risk_per_trade']:.1%}, PT={best_sharpe['profit_target_r']:.1f}R, SL={best_sharpe['stop_loss_r']:.1f}R")
print(f"Results: Return={best_sharpe['total_return']:.2%}, Sharpe={best_sharpe['sharpe_ratio']:.3f}, MDD={best_sharpe['mdd_pct']:.2%}")
